# Estrazione Codice Penale da Normattiva

Notebook per estrarre tutti gli articoli del Codice Penale da Normattiva in CSV e validare il risultato.

Output principale: `src/data/statutes/codice_penale_normattiva.csv`.

Schema output:
- `articolo`
- `titolo`
- `testo`
- `reference` (lista JSON di riferimenti interni al codice)
- `external_reference` (lista JSON di riferimenti esterni)
- `libro_codice_penale`

Integrazione Neo4j:
- `src/db/db_orchestrator.py` legge `reference` e `external_reference` in ingestione.
- I riferimenti interni in `reference` sono usati per creare relazioni `(:Statute)-[:CITES]->(:Statute)` intra-codice.

- I riferimenti sono normalizzati con correzioni di suffisso e whitelist interna: se un articolo non esiste nel codice, viene spostato in `external_reference`/`external_references`.


In [4]:
from __future__ import annotations

import csv
import json
import html
import http.cookiejar
import re
import urllib.request
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

BASE_NORMATTIVA_URL = "https://www.normattiva.it/uri-res/N2Ls?urn:nir:stato:regio.decreto:1930-10-19;1398"
OUTPUT_CSV_PATH = Path("../src/data/statutes/codice_penale_normattiva.csv")
RAW_DEBUG_DIR = Path("./tmp_normattiva_cp")
SAVE_RAW_HTML = False
INCLUDE_UPDATES = False

ARTICLE_LINK_RE = re.compile(
    r"onclick=\"return showArticle\('(/atto/caricaArticolo\?[^']+)'\s*,\s*this\);\"[^>]*class=\"numero_articolo\">\s*art\.\s*([^<]+?)\s*</a>",
    re.IGNORECASE,
)
ART_REF_RE = re.compile(r"\bart(?:t|icolo)?\.?\s*(\d+(?:-[a-z]+)*(?:\.\d+)?)", re.IGNORECASE)
EXTERNAL_REFERENCE_MARKERS = (
    "c.p.p",
    "codice di procedura penale",
    "c.p.c",
    "codice di procedura civile",
    "c.c.",
    "codice civile",
    "cost.",
    "costituzione",
    "decreto",
    "d.lgs",
    "d.l.",
    "dpr",
    "legge",
)


@dataclass
class ArticleEntry:
    articolo: str
    titolo: str
    testo: str
    reference: str
    external_reference: str
    libro_codice_penale: str


def _build_opener() -> urllib.request.OpenerDirector:
    jar = http.cookiejar.CookieJar()
    opener = urllib.request.build_opener(urllib.request.HTTPCookieProcessor(jar))
    opener.addheaders = [("User-Agent", "Mozilla/5.0")]
    return opener


def _normalize_article_label(raw_label: str) -> str:
    label = raw_label.strip().lower()
    label = label.replace("‑", "-").replace("–", "-").replace("—", "-")
    label = re.sub(r"\s+", "-", label)
    label = label.strip(".")
    return label


_SUFFIX_ORDER = {
    "bis": 1,
    "ter": 2,
    "quater": 3,
    "quinquies": 4,
    "sexies": 5,
    "septies": 6,
    "octies": 7,
    "novies": 8,
    "decies": 9,
    "undecies": 10,
    "duodecies": 11,
    "terdecies": 12,
    "quaterdecies": 13,
    "quinquiesdecies": 14,
    "sexiesdecies": 15,
    "septiesdecies": 16,
    "duodevicies": 17,
    "vicies": 18,
}


def _token_sort_key(token: str) -> tuple[int, int | str]:
    if token.isdigit():
        return (2, int(token))
    order = _SUFFIX_ORDER.get(token)
    if order is not None:
        return (1, order)
    return (3, token)


def _article_sort_key(label: str) -> tuple[int, tuple[tuple[int, int | str], ...], str]:
    normalized = _normalize_article_label(label)
    m = re.match(r"^(\d+)(.*)$", normalized)
    if not m:
        return (10**9, tuple(), normalized)
    base = int(m.group(1))
    rest = m.group(2).strip("-./")
    if not rest:
        return (base, tuple(), "")
    tokens = [t for t in re.split(r"[-./]", rest) if t]
    token_keys = tuple(_token_sort_key(t) for t in tokens)
    return (base, token_keys, rest)



def _html_to_lines(fragment: str) -> list[str]:
    text = re.sub(r"(?is)<script.*?>.*?</script>", " ", fragment)
    text = re.sub(r"(?is)<style.*?>.*?</style>", " ", text)
    text = re.sub(r"(?i)<br\s*/?>", "\n", text)
    text = re.sub(r"(?i)</div>|</p>|</li>|</h\d>", "\n", text)
    text = re.sub(r"(?is)<[^>]+>", " ", text)
    text = html.unescape(text)
    lines = [ln.strip() for ln in text.splitlines()]
    return [ln for ln in lines if ln]


def _clean_title(raw_title: str) -> str:
    title = raw_title.strip()
    title = re.sub(r"^[\(\)\.\s]+", "", title)
    title = re.sub(r"[\(\)\.\s]+$", "", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


def _infer_libro(articolo: str) -> str:
    m = re.match(r"^(\d+)", articolo)
    if not m:
        return ""
    n = int(m.group(1))
    if n <= 240:
        return "Libro I"
    if n <= 649:
        return "Libro II"
    return "Libro III"


REF_SUFFIX_CORRECTIONS = {
    "nonies": "novies",
    "sexiesdecies": "sexdecies",
}


def _normalize_reference_suffixes(token: str) -> str:
    parts = token.split("-")
    if len(parts) <= 1:
        return token
    fixed = [parts[0]]
    for part in parts[1:]:
        fixed.append(REF_SUFFIX_CORRECTIONS.get(part, part))
    return "-".join(fixed)


def _normalize_reference_article(raw_ref: str) -> str:
    token = raw_ref.strip().lower()
    token = token.replace("‑", "-").replace("–", "-").replace("—", "-")
    m = re.search(r"(\d+(?:-[a-z]+)*(?:\.\d+)?)", token)
    if not m:
        return ""
    return _normalize_reference_suffixes(m.group(1))


def _extract_references(text: str, valid_internal_refs: set[str] | None = None) -> tuple[list[str], list[str]]:
    if not text:
        return [], []

    internal: set[str] = set()
    external: set[str] = set()

    for match in ART_REF_RE.finditer(text):
        ref = _normalize_reference_article(match.group(1))
        if not ref:
            continue

        w_start = max(0, match.start() - 64)
        w_end = min(len(text), match.end() + 96)
        window = text[w_start:w_end].lower()

        is_internal_cp = ("c.p." in window) or ("codice penale" in window)
        is_external = any(marker in window for marker in EXTERNAL_REFERENCE_MARKERS)

        if is_external and not is_internal_cp:
            label = "Art. " + ref
            if "c.c." in window or "codice civile" in window:
                label += " c.c."
            elif "c.p.p" in window or "codice di procedura penale" in window:
                label += " c.p.p."
            elif "c.p.c" in window or "codice di procedura civile" in window:
                label += " c.p.c."
            elif "cost" in window:
                label += " Cost."
            external.add(label)
            continue

        if valid_internal_refs is not None and ref not in valid_internal_refs:
            external.add("Art. " + ref)
            continue

        internal.add(ref)

    return sorted(internal, key=_article_sort_key), sorted(external)


def _extract_title_and_body(lines: list[str], articolo: str) -> tuple[str, str]:
    if not lines:
        return "", ""

    heading_pattern = re.escape(articolo).replace("-", r"[-\s]?")
    art_re = re.compile(
        rf"^Art\.\s*{heading_pattern}\.?",
        re.IGNORECASE,
    )

    art_idx = 0
    for i, line in enumerate(lines):
        if art_re.search(line):
            art_idx = i
            break

    content = lines[art_idx + 1 :] if art_idx + 1 < len(lines) else []
    if not content:
        return "", ""

    raw_title = content[0]
    title = _clean_title(raw_title)

    # Se la prima riga è chiaramente già testo normativo, non trattarla come titolo
    raw_title_stripped = raw_title.strip()
    looks_like_body = (
        not raw_title_stripped.startswith("(")
        and bool(re.match(r"^(Chiunque|Fuori|Quando|Qualora|La|Le|Il|I|Non|Nei|Nel|Se)\b", title, re.IGNORECASE))
        and len(title) > 60
    )

    if looks_like_body:
        title = ""
        body_lines = content
    else:
        body_lines = content[1:]

    if not INCLUDE_UPDATES:
        cut_idx = None
        for i, ln in enumerate(body_lines):
            if ln.upper().startswith("AGGIORNAMENTO") or ln.startswith("------------"):
                cut_idx = i
                break
        if cut_idx is not None:
            body_lines = body_lines[:cut_idx]

    body = " ".join(body_lines).strip()
    body = re.sub(r"\s+", " ", body).strip()
    body = re.sub(r"^\.\s*", "", body)
    return title, body


def extract_codice_penale() -> list[ArticleEntry]:
    opener = _build_opener()
    root_html = opener.open(BASE_NORMATTIVA_URL, timeout=60).read().decode("utf-8", "ignore")

    if SAVE_RAW_HTML:
        RAW_DEBUG_DIR.mkdir(parents=True, exist_ok=True)
        (RAW_DEBUG_DIR / "root.html").write_text(root_html, encoding="utf-8")

    # Articolo -> path AJAX. Se duplicato, mantiene il primo occorrente.
    article_paths: dict[str, str] = {}
    for m in ARTICLE_LINK_RE.finditer(root_html):
        ajax_path = html.unescape(m.group(1))
        label = _normalize_article_label(html.unescape(m.group(2)))
        if label not in article_paths:
            article_paths[label] = ajax_path

    valid_internal_refs = set(article_paths.keys())

    rows: list[ArticleEntry] = []
    for idx, (articolo, ajax_path) in enumerate(sorted(article_paths.items(), key=lambda x: _article_sort_key(x[0])), start=1):
        req = urllib.request.Request(
            "https://www.normattiva.it" + ajax_path,
            headers={
                "User-Agent": "Mozilla/5.0",
                "X-Requested-With": "XMLHttpRequest",
                "Referer": BASE_NORMATTIVA_URL,
                "Accept": "*/*",
            },
        )
        payload = opener.open(req, timeout=60).read().decode("utf-8", "ignore")

        if SAVE_RAW_HTML and idx <= 20:
            (RAW_DEBUG_DIR / f"article_{idx:04d}_{articolo}.html").write_text(payload, encoding="utf-8")

        start = payload.find('<div class="bodyTesto">')
        end = payload.find('<div class="d-flex justify-content-between', start)
        if start == -1 or end == -1:
            continue

        body_fragment = payload[start:end]
        lines = _html_to_lines(body_fragment)
        titolo, testo = _extract_title_and_body(lines, articolo)

        # fallback: se il corpo è vuoto, usa tutto dopo l'header articolo
        if not testo and lines:
            if len(lines) > 1:
                testo = " ".join(lines[1:]).strip()
                testo = re.sub(r"\s+", " ", testo)
                testo = re.sub(r"^\.\s*", "", testo)

        internal_refs, external_refs = _extract_references(testo, valid_internal_refs=valid_internal_refs)

        rows.append(
            ArticleEntry(
                articolo=articolo,
                titolo=titolo,
                testo=testo,
                reference=json.dumps(internal_refs, ensure_ascii=False),
                external_reference=json.dumps(external_refs, ensure_ascii=False),
                libro_codice_penale=_infer_libro(articolo),
            )
        )

    return rows


def save_csv(rows: list[ArticleEntry], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(
            f,
            fieldnames=[
                "articolo",
                "titolo",
                "testo",
                "reference",
                "external_reference",
                "libro_codice_penale",
            ],
        )
        w.writeheader()
        sorted_rows = sorted(rows, key=lambda r: _article_sort_key(r.articolo))
        for row in sorted_rows:
            w.writerow(row.__dict__)


rows = extract_codice_penale()
save_csv(rows, OUTPUT_CSV_PATH)

print(f"Estrazione completata: {len(rows)} articoli")
print(f"CSV salvato in: {OUTPUT_CSV_PATH.resolve()}")
print(f"Timestamp UTC: {datetime.now(timezone.utc).isoformat()}")

for sample in ["43", "54", "624", "624-bis", "625", "639-ter", "649-bis"]:
    item = next((r for r in rows if r.articolo == sample), None)
    if item:
        print(f"- Art. {sample}: titolo='{item.titolo[:80]}' | incipit='{item.testo[:120]}'")


Estrazione completata: 985 articoli
CSV salvato in: /Users/l.catello/Library/Mobile Documents/com~apple~CloudDocs/Magistrale Ingegneria Informatica/Tesi/LexCausa/src/data/statutes/codice_penale_normattiva.csv
Timestamp UTC: 2026-02-20T21:57:29.608191+00:00
- Art. 43: titolo='Elemento psicologico del reato' | incipit='Il delitto: è doloso, o secondo l'intenzione, quando l'evento dannoso o pericoloso, che è il risultato dell'azione od om'
- Art. 54: titolo='Stato di necessità' | incipit='Non è punibile chi ha commesso il fatto per esservi stato costretto dalla necessità di salvare sé od altri dal pericolo '
- Art. 624: titolo='Furto' | incipit='Chiunque s'impossessa della cosa mobile altrui, sottraendola a chi la detiene, al fine di trarne profitto per sé o per a'
- Art. 624-bis: titolo='Furto in abitazione e furto con strappo' | incipit='Chiunque si impossessa della cosa mobile altrui, sottraendola a chi la detiene, al fine di trarne profitto per sé o per '
- Art. 625: titolo='Circostan

In [5]:
import csv
import re
from collections import Counter

OUT = OUTPUT_CSV_PATH
with OUT.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print("Righe:", len(rows))
print("Colonne:", rows[0].keys())

arts = [(r.get("articolo") or "").strip() for r in rows]
dup = [(k, v) for k, v in Counter(arts).items() if v > 1]
print("Duplicati articolo:", len(dup))

bad_art = [a for a in arts if not re.match(r"^\d+(?:-[a-z]+(?:\.\d+)?)?$", a)]
print("Articolo formato anomalo:", len(bad_art))
if bad_art:
    print("Esempi:", bad_art[:20])

missing_title = sum(1 for r in rows if not (r.get("titolo") or "").strip())
missing_text = sum(1 for r in rows if not (r.get("testo") or "").strip())
print("Titolo vuoto:", missing_title)
print("Testo vuoto:", missing_text)

leading_dot = sum(1 for r in rows if (r.get("testo") or "").lstrip().startswith("."))
paren_title = sum(
    1
    for r in rows
    if (r.get("titolo") or "").lstrip().startswith("((")
    or (r.get("titolo") or "").lstrip().startswith("( (")
)
print("Testo con leading dot:", leading_dot)
print("Titolo con artefatto parentesi:", paren_title)

for a in ["43", "54", "59", "61", "624", "624-bis", "625", "639-ter", "649-bis", "609-novies"]:
    row = next((r for r in rows if (r.get("articolo") or "").strip() == a), None)
    print(f"\nArt. {a}:", "FOUND" if row else "MISSING")
    if row:
        print("  titolo:", (row.get("titolo") or "")[:100])
        print("  incipit:", (row.get("testo") or "")[:130])


Righe: 985
Colonne: dict_keys(['articolo', 'titolo', 'testo', 'reference', 'external_reference', 'libro_codice_penale'])
Duplicati articolo: 0
Articolo formato anomalo: 0
Titolo vuoto: 0
Testo vuoto: 0
Testo con leading dot: 0
Titolo con artefatto parentesi: 0

Art. 43: FOUND
  titolo: Elemento psicologico del reato
  incipit: Il delitto: è doloso, o secondo l'intenzione, quando l'evento dannoso o pericoloso, che è il risultato dell'azione od omissione e 

Art. 54: FOUND
  titolo: Stato di necessità
  incipit: Non è punibile chi ha commesso il fatto per esservi stato costretto dalla necessità di salvare sé od altri dal pericolo attuale di

Art. 59: FOUND
  titolo: Circostanze non conosciute o erroneamente supposte
  incipit: ((Le circostanze che attenuano o escludono la pena sono valutate a favore dell'agente anche se da lui non conosciute, o da lui per

Art. 61: FOUND
  titolo: Circostanze aggravanti comuni
  incipit: Aggravano il reato, quando non ne sono elementi costitutivi o circo